# ДЗ 1.2: Spark RDD + DataFrame API

Данные лежат в HDFS после ДЗ 1.1 (`/hw1/input`), там же результаты MapReduce (`/hw1/wordcount`, `/hw1/top5`) — с ними и сверяемся.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import heapq

spark = SparkSession.builder.appName("hw1_2").getOrCreate()
sc = spark.sparkContext

In [2]:
! hdfs dfs -ls /hw1/input

Found 2 items
-rw-r--r--   1 hadoop users   36443383 2026-10-01 16:33 /hw1/input/Poseschenia_sai_774_tov.csv
-rw-r--r--   1 hadoop users    4047392 2026-10-01 16:33 /hw1/input/bible.txt


## Задание 1

### RDD

In [3]:
words_rdd = (
    sc.textFile("hdfs:///hw1/input/bible.txt")
    .flatMap(lambda line: line.split())
    .filter(lambda w: len(w) > 4)
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
)

words_rdd.sortBy(lambda x: -x[1]).take(10)

[('shall', 9658),
 ('which', 4119),
 ('their', 3801),
 ('there', 1799),
 ('before', 1701),
 ('against', 1587),
 ('shalt', 1586),
 ('children', 1559),
 ('said,', 1556),
 ('them,', 1549)]

### DataFrame API

In [4]:
words_df = (
    spark.read.text("hdfs:///hw1/input/bible.txt")
    .select(F.explode(F.split("value", r"\s+")).alias("word"))
    .filter(F.length("word") > 4)
    .groupBy("word")
    .count()
)

words_df.orderBy(F.desc("count")).show(10)

+--------+-----+
|    word|count|
+--------+-----+
|   shall| 9658|
|   which| 4119|
|   their| 3801|
|   there| 1799|
|  before| 1701|
| against| 1587|
|   shalt| 1586|
|children| 1559|
|   said,| 1556|
|   them,| 1549|
+--------+-----+
only showing top 10 rows



### Сверка с MR

In [5]:
words_mr = (
    sc.textFile("hdfs:///hw1/wordcount")
    .map(lambda line: line.split("\t"))
    .map(lambda p: (p[0], int(p[1])))
    .collectAsMap()
)
words_rdd_res = words_rdd.collectAsMap()
words_df_res = {row["word"]: row["count"] for row in words_df.collect()}

print("слов: MR", len(words_mr), "RDD", len(words_rdd_res), "DF", len(words_df_res))
print("MR == RDD:", words_mr == words_rdd_res)
print("MR == DF:", words_mr == words_df_res)

слов: MR 26280 RDD 26280 DF 26280
MR == RDD: True
MR == DF: True


Все три решения совпадают.

## Задание 2

### RDD

Сначала считаем посещения по ключу (дата, сайт), потом для каждой даты оставляем топ-5 через `reduceByKey`:
при слиянии храним не больше 5 элементов, поэтому памяти O(1) на ключ (без `groupByKey` и списков всех сайтов).

In [6]:
def parse(line):
    site, dt = line.strip().split(";")
    return (dt[:10], site), 1


def top5(a, b):
    return heapq.nlargest(5, a + b)


top5_rdd = (
    sc.textFile("hdfs:///hw1/input/Poseschenia_sai_774_tov.csv")
    .map(parse)
    .reduceByKey(lambda a, b: a + b)
    .map(lambda x: (x[0][0], [(x[1], x[0][1])]))
    .reduceByKey(top5)
    .flatMap(lambda x: [(x[0], site, cnt) for cnt, site in x[1]])
)

for row in sorted(top5_rdd.collect(), key=lambda r: (r[0], -r[2])):
    print(*row, sep="\t")

2024-05-26	https://gonzales-bautista.com/	335
2024-05-26	http://smith.com/	235
2024-05-26	https://www.smith.com/	221
2024-05-26	https://smith.com/	212
2024-05-26	http://www.smith.com/	212
2024-05-27	https://gonzales-bautista.com/	376
2024-05-27	https://www.smith.com/	270
2024-05-27	https://smith.com/	236
2024-05-27	http://smith.com/	215
2024-05-27	http://www.smith.com/	208
2024-05-28	https://gonzales-bautista.com/	368
2024-05-28	https://smith.com/	256
2024-05-28	https://www.smith.com/	251
2024-05-28	http://smith.com/	224
2024-05-28	http://www.smith.com/	204
2024-05-29	https://gonzales-bautista.com/	402
2024-05-29	https://www.smith.com/	242
2024-05-29	http://www.smith.com/	223
2024-05-29	https://smith.com/	220
2024-05-29	http://smith.com/	206
2024-05-30	https://gonzales-bautista.com/	353
2024-05-30	https://smith.com/	246
2024-05-30	https://www.smith.com/	239
2024-05-30	http://smith.com/	229
2024-05-30	http://www.smith.com/	225
2024-05-31	https://gonzales-bautista.com/	374
2024-05-31	htt

### DataFrame API

In [7]:
visits = (
    spark.read.csv("hdfs:///hw1/input/Poseschenia_sai_774_tov.csv", sep=";")
    .toDF("site", "dt")
    .withColumn("date", F.substring("dt", 1, 10))
)

w = Window.partitionBy("date").orderBy(F.desc("count"), F.desc("site"))

top5_df = (
    visits.groupBy("date", "site").count()
    .withColumn("rank", F.row_number().over(w))
    .filter("rank <= 5")
    .select("date", "site", "count")
)

top5_df.orderBy("date", F.desc("count")).show(40, truncate=False)

+----------+------------------------------+-----+
|date      |site                          |count|
+----------+------------------------------+-----+
|2024-05-26|https://gonzales-bautista.com/|335  |
|2024-05-26|http://smith.com/             |235  |
|2024-05-26|https://www.smith.com/        |221  |
|2024-05-26|https://smith.com/            |212  |
|2024-05-26|http://www.smith.com/         |212  |
|2024-05-27|https://gonzales-bautista.com/|376  |
|2024-05-27|https://www.smith.com/        |270  |
|2024-05-27|https://smith.com/            |236  |
|2024-05-27|http://smith.com/             |215  |
|2024-05-27|http://www.smith.com/         |208  |
|2024-05-28|https://gonzales-bautista.com/|368  |
|2024-05-28|https://smith.com/            |256  |
|2024-05-28|https://www.smith.com/        |251  |
|2024-05-28|http://smith.com/             |224  |
|2024-05-28|http://www.smith.com/         |204  |
|2024-05-29|https://gonzales-bautista.com/|402  |
|2024-05-29|https://www.smith.com/        |242  |


### Сверка с MR

При одинаковом числе посещений решения могут выбрать разные сайты (порядок при равенстве в ТЗ не задан),
поэтому сравниваю пары (дата, число посещений) — они должны совпадать всегда. И отдельно показываю строки, которые отличаются.

In [8]:
top5_mr = (
    sc.textFile("hdfs:///hw1/top5")
    .map(lambda line: line.split("\t"))
    .map(lambda p: (p[0], p[1], int(p[2])))
    .collect()
)
top5_rdd_res = top5_rdd.collect()
top5_df_res = [tuple(row) for row in top5_df.collect()]

def date_counts(rows):
    return sorted((date, cnt) for date, site, cnt in rows)

print("MR == RDD:", date_counts(top5_mr) == date_counts(top5_rdd_res))
print("MR == DF:", date_counts(top5_mr) == date_counts(top5_df_res))
print("RDD == DF (вместе с сайтами):", sorted(top5_rdd_res) == sorted(top5_df_res))
print("есть в MR, но нет в Spark:", set(top5_mr) - set(top5_rdd_res))

MR == RDD: True
MR == DF: True
RDD == DF (вместе с сайтами): True
есть в MR, но нет в Spark: {('2024-06-02', 'http://www.johnson.com/', 5), ('2024-06-02', 'http://miller.com/', 5)}


In [9]:
spark.stop()